<a href="https://colab.research.google.com/github/LilianaHerrera/Maestria/blob/main/Clase_Cierre_Analista_IA_Spark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analista de datos con IA generativa y Spark
### Clase de cierre — Computación en la Nube · Maestría en Ciencia de Datos

En esta clase se construye un **asistente de datos**: el usuario hace una pregunta en lenguaje natural (por ejemplo, *"¿cuál es la tarifa media por zona?"*) y un modelo de lenguaje (LLM) **escribe el código de análisis en PySpark**, que luego se ejecuta sobre los datos. Al final se publica como una aplicación web.

**Por qué cierra el curso:** une las dos mitades del semestre. El procesamiento con **Spark** (clase 1) y el **despliegue de una app con un modelo** (clase 3) se combinan con la técnica más actual: que el LLM **genere y ejecute el análisis** (text-to-code), la base de los asistentes de datos modernos.

| Herramienta | Qué es | Para qué se usa aquí |
|---|---|---|
| **PySpark** | Motor de procesamiento de Big Data | Ejecutar el análisis sobre los datos (escala a datos grandes) |
| **Gemini (Google AI Studio)** | LLM con capa gratuita, sin tarjeta ni GCP | Traducir la pregunta en lenguaje natural a código PySpark |
| **Gradio** | Interfaz web | El cuadro donde se escribe la pregunta y se ve el resultado |
| **Hugging Face Spaces** | Despliegue gratuito | Publicar la aplicación en internet |

> El dataset es el de taxis del curso, ampliado con columnas para analizar (propina, tipo de pago, zona).

## Paso 0 — Instalación de dependencias

In [1]:
# PASO 0 — Instalación de dependencias (versiones fijas y compatibles entre sí).
#   pyspark          -> motor de procesamiento de datos
#   gradio           -> interfaz web
#   openai           -> cliente para llamar al modelo GLM de Z.ai (API compatible con OpenAI)
#   huggingface_hub  -> para publicar la app en Hugging Face
!pip install -q \
  pyspark==4.1.2 \
  gradio==6.17.3 \
  openai \
  starlette==1.3.0 \
  fastapi==0.136.3 \
  huggingface_hub==1.19.0

print("Dependencias instaladas")

Dependencias instaladas


In [2]:
# Verificación de versiones (celda opcional)
# Sirve para confirmar, después de instalar y reiniciar la sesión, que quedaron
# las versiones correctas de starlette y gradio. Si no coinciden, la aplicación
# (celda 4b) puede fallar al iniciarse.
import starlette, gradio
print("starlette", starlette.__version__, "| gradio", gradio.__version__)
# Resultado esperado: starlette 1.3.0 | gradio 6.17.3

starlette 1.3.0 | gradio 6.17.3


## Paso 1 — Spark y los datos
Se inicia Spark y se cargan los datos. Aquí se generan datos de taxis con la misma estructura del curso.
PARA LA ACTIVIDAD: para usar datos reales, reemplazar `_crear_datos()` por `spark.read.csv("archivo.csv", header=True, inferSchema=True)`.

In [3]:
# PASO 1 — Iniciar Spark
# Spark es el motor que procesa los datos. Aquí se crea la "sesión de Spark",
# que es el punto de entrada para trabajar con DataFrames.

import numpy as np, pandas as pd
from pyspark.sql import SparkSession, functions as F   # F = funciones de Spark (avg, sum, etc.)

# Se construye la sesión de Spark:
#   appName("analista")        -> nombre de la aplicación (aparece en los registros)
#   master("local[*]")         -> ejecuta en esta misma máquina usando TODOS los núcleos ([*])
#   spark.ui.enabled = false   -> desactiva el panel web de Spark (no se necesita en Colab)
#   getOrCreate()              -> crea la sesión, o reutiliza una si ya existe
spark = (SparkSession.builder.appName("analista").master("local[*]")
         .config("spark.ui.enabled", "false").getOrCreate())

# Se baja el nivel de mensajes: solo se muestran errores, no las advertencias de Spark
spark.sparkContext.setLogLevel("ERROR")

In [4]:
# PASO 1 (continuación) — Crear los datos de ejemplo
# Esta función fabrica viajes de taxi "sintéticos" (inventados pero realistas) para
# tener datos sobre los cuales preguntar. Cada fila será un viaje; cada columna, un dato.

def _crear_datos(n=200000):
    """Genera n viajes de taxi de ejemplo (con columnas extra para poder analizar)."""

    # Generador de números aleatorios con semilla fija (42): garantiza que los datos
    # sean SIEMPRE los mismos cada vez que se ejecuta (resultados reproducibles).
    rng = np.random.default_rng(42)

    # --- Variables de cada viaje ---
    # Distancia en millas. La distribución 'gamma' produce muchos viajes cortos y pocos
    # largos (como en la realidad). .clip(0.3, 30) limita el rango; .round(2) deja 2 decimales.
    dist = rng.gamma(2, 1.4, n).clip(0.3, 30).round(2)

    hour = rng.integers(0, 24, n)                  # hora de recogida: entero de 0 a 23
    wk   = rng.binomial(1, 2/7, n)                 # ¿fin de semana? 1 = sí, 0 = no (2 de cada 7 días)
    pax  = rng.choice([1,1,1,2,2,3,4,5,6], n)      # pasajeros (el 1 se repite: lo más común es 1)

    # Velocidad promedio en millas/hora, con variación por el tráfico (alrededor de 18 mph)
    sp   = (18 + rng.normal(0, 3, n)).clip(8, 30)
    # Duración del viaje en minutos = (distancia / velocidad) * 60, más una pequeña variación
    dur  = (dist/sp*60 + rng.normal(0, 1.5, n)).clip(1, 120).round(1)

    # Marcas de momento del día (sirven para el recargo de la tarifa):
    rush = (((hour>=7)&(hour<=9)) | ((hour>=17)&(hour<=19))) & (wk==0)  # hora pico (solo entre semana)
    night= (hour>=0)&(hour<=5)                                          # madrugada

    # Tarifa estilo Nueva York: base + cobro por milla + cobro por minuto
    # + recargo en hora pico + recargo nocturno - descuento de fin de semana
    # + un poco de ruido aleatorio. .clip(3, None) asegura que nunca baje de 3 dólares.
    fare = (3 + 2.5*dist + 0.4*dur + rush*3 + night*1 - wk*0.5 + rng.normal(0,1,n)).clip(3, None).round(2)

    pago = rng.choice(["tarjeta","efectivo"], n, p=[0.7, 0.3])  # tipo de pago (70% tarjeta)
    # Propina: solo si pagó con tarjeta, entre 0% y 25% de la tarifa; en efectivo, 0
    tip  = np.where(pago=="tarjeta", fare*rng.uniform(0,0.25,n), 0).round(2)
    # Zona de recogida (con distintas probabilidades, Manhattan es la más frecuente)
    zona = rng.choice(["Manhattan","Brooklyn","Queens","Bronx","Aeropuerto"], n, p=[0.45,0.2,0.15,0.1,0.1])

    # Se arma una tabla de pandas con todas las columnas
    pdf = pd.DataFrame({"trip_distance":dist,"trip_duration_min":dur,"passenger_count":pax,
                        "pickup_hour":hour,"is_weekend":wk,"fare_amount":fare,
                        "tip_amount":tip,"payment_type":pago,"pickup_zone":zona})

    # Se convierte la tabla de pandas en un DataFrame de Spark y se devuelve
    return spark.createDataFrame(pdf)

In [5]:
df = _crear_datos()                                            # PARA LA ACTIVIDAD: cargar datos reales aquí
ESQUEMA = ", ".join([f"{c} ({t})" for c, t in df.dtypes])     # texto con las columnas y sus tipos

print("Filas:", df.count(), "| Columnas:", len(df.columns))
print("Esquema:", ESQUEMA)
df.show(5)

Filas: 200000 | Columnas: 9
Esquema: trip_distance (double), trip_duration_min (double), passenger_count (bigint), pickup_hour (bigint), is_weekend (bigint), fare_amount (double), tip_amount (double), payment_type (string), pickup_zone (string)
+-------------+-----------------+---------------+-----------+----------+-----------+----------+------------+-----------+
|trip_distance|trip_duration_min|passenger_count|pickup_hour|is_weekend|fare_amount|tip_amount|payment_type|pickup_zone|
+-------------+-----------------+---------------+-----------+----------+-----------+----------+------------+-----------+
|         2.93|              7.7|              6|         11|         1|      14.23|      3.42|     tarjeta|  Manhattan|
|         3.97|             10.0|              3|         21|         1|      16.66|      1.73|     tarjeta|  Manhattan|
|         2.57|              7.5|              3|          4|         0|      12.46|       0.0|    efectivo|  Manhattan|
|          2.3|              

## Paso 2 — El LLM que escribe el código (GLM de Z.ai)

El LLM recibe la pregunta y el esquema de los datos, y devuelve el código PySpark que la responde.

Se necesita una **API key gratuita** de Z.ai (no pide tarjeta de crédito):

1. Entrar a **z.ai** e iniciar sesión (con correo o cuenta de Google).
2. Ir a la página **API Keys** y crear una clave con **"Create API key"**.
3. Copiar la clave (formato `{API Key ID}.{secret}`, con un punto en medio) y pegarla cuando la celda la solicite.

> El detalle paso a paso está en el **Manual_ZAI_API_Key**.

In [6]:
# PASO 2 — Conectar el modelo de lenguaje (GLM de Z.ai)
# Librerías necesarias para este paso:
import os                          # para guardar la clave en una "variable de entorno"
from getpass import getpass        # para pedir la clave de forma segura (no se ve al escribirla)
from openai import OpenAI          # cliente para hablar con el modelo (Z.ai es compatible con OpenAI)

# Clave de Z.ai (se crea gratis en z.ai -> perfil -> API Keys). Se ingresa de forma segura.
os.environ["ZAI_API_KEY"] = getpass("API key de Z.ai (GLM): ")

# Z.ai es compatible con OpenAI: se usa el cliente de OpenAI apuntando a su servidor.
cliente = OpenAI(api_key=os.environ["ZAI_API_KEY"], base_url="https://api.z.ai/api/paas/v4")
MODELO_LLM = "glm-4.5-flash"   # variante Flash (capa gratuita); puede ajustarse (ej. "glm-4.7-flash")

# Instrucciones para que el modelo devuelva SOLO código PySpark.
INSTRUCCION = f"""Eres un analista de datos experto en PySpark.
Existe un DataFrame de Spark llamado df con estas columnas: {ESQUEMA}.
Escribe codigo PySpark que responda la pregunta del usuario.
Reglas estrictas:
- Usa unicamente df, spark y F (pyspark.sql.functions). No leas ni escribas archivos.
- Asigna SIEMPRE el resultado final a una variable llamada resultado.
- Si el resultado es una tabla, deja resultado como un DataFrame de Spark (no uses .show()).
- Devuelve SOLO el codigo Python, sin explicaciones ni bloques de markdown."""

def generar_codigo(pregunta):
    """Pide al modelo (GLM) el código PySpark para la pregunta y limpia la respuesta."""
    r = cliente.chat.completions.create(
        model=MODELO_LLM,
        messages=[
            {"role": "system", "content": INSTRUCCION},
            {"role": "user", "content": pregunta},
        ],
    )
    codigo = r.choices[0].message.content.strip()
    if codigo.startswith("```"):                 # quita las comillas de markdown si vinieran
        codigo = codigo.strip("`")
        if codigo.lstrip().startswith("python"):
            codigo = codigo.lstrip()[len("python"):]
    return codigo.strip()

# Prueba: se le pide el código para una pregunta y se muestra lo que escribió el modelo
print(generar_codigo("¿Cuál es la tarifa media por tipo de pago?"))




API key de Z.ai (GLM): ··········
from pyspark.sql import functions as F

resultado = df.groupBy("payment_type").agg(F.mean("fare_amount").alias("fare_amount_mean"))


## Paso 3 — Ejecutar el código generado
El código que escribió el LLM se ejecuta sobre los datos. **Importante:** se ejecuta código generado por la IA; por eso corre en un entorno controlado, solo con `df`, `spark` y `F`, y sobre datos propios.

## Paso 4 — La aplicación (interfaz web)
Se guarda toda la lógica en `app.py`: Spark, el LLM, la ejecución y la interfaz de Gradio. La API key se lee de la variable de entorno `GEMINI_API_KEY` (en el despliegue se configura como un "secret").

In [7]:
%%writefile app.py
# Esta celda crea el archivo app.py: la aplicación que luego se despliega en Hugging Face.
# (%%writefile guarda en un archivo todo lo que está debajo; no lo ejecuta aquí.)
# Esta primera parte importa las librerías y arranca Spark.
"""
Analista de datos con IA generativa y Spark.
El usuario escribe una pregunta en lenguaje natural; el modelo GLM (Z.ai) traduce esa
pregunta a código PySpark, el código se ejecuta sobre los datos y se muestra el resultado.
La interfaz es web (Gradio). La clave de Z.ai se lee de la variable de entorno ZAI_API_KEY.
"""
import os, warnings
warnings.filterwarnings("ignore")     # silencia advertencias para que la salida quede limpia

import numpy as np, pandas as pd      # cálculo numérico y tablas de datos

import matplotlib
matplotlib.use("Agg")                 # dibuja gráficos sin necesidad de pantalla (la app corre en un servidor)
import matplotlib.pyplot as plt

import gradio as gr                   # crea la interfaz web (el formulario y los resultados)
from openai import OpenAI            # cliente para llamar al modelo GLM (Z.ai usa el formato de OpenAI)
from pyspark.sql import SparkSession, functions as F   # Spark; F agrupa funciones como avg() o sum()

# ---------- 1) Iniciar Spark ----------
# Se crea la sesión de Spark, que procesará los datos dentro de la aplicación.
#   master("local[*]")  -> usa esta misma máquina con todos sus núcleos
#   spark.ui.enabled=false -> no levanta el panel web de Spark (no hace falta)
spark = (SparkSession.builder.appName("analista").master("local[*]")
         .config("spark.ui.enabled", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")   # muestra solo errores, no las advertencias de Spark

def _crear_datos(n):
    """Genera un dataset de viajes de taxi (mismas variables del curso, con columnas extra para analizar)."""
    rng = np.random.default_rng(42)
    dist = rng.gamma(2, 1.4, n).clip(0.3, 30).round(2)
    hour = rng.integers(0, 24, n); wk = rng.binomial(1, 2/7, n)
    pax = rng.choice([1,1,1,2,2,3,4,5,6], n)
    sp = (18 + rng.normal(0, 3, n)).clip(8, 30)
    dur = (dist/sp*60 + rng.normal(0, 1.5, n)).clip(1, 120).round(1)
    rush = (((hour>=7)&(hour<=9)) | ((hour>=17)&(hour<=19))) & (wk==0)
    night = (hour>=0)&(hour<=5)
    fare = (3 + 2.5*dist + 0.4*dur + rush*3 + night*1 - wk*0.5 + rng.normal(0,1,n)).clip(3, None).round(2)
    pago = rng.choice(["tarjeta", "efectivo"], n, p=[0.7, 0.3])
    tip = np.where(pago=="tarjeta", fare*rng.uniform(0, 0.25, n), 0).round(2)
    zona = rng.choice(["Manhattan","Brooklyn","Queens","Bronx","Aeropuerto"], n, p=[0.45,0.2,0.15,0.1,0.1])
    pdf = pd.DataFrame({"trip_distance":dist, "trip_duration_min":dur, "passenger_count":pax,
                        "pickup_hour":hour, "is_weekend":wk, "fare_amount":fare,
                        "tip_amount":tip, "payment_type":pago, "pickup_zone":zona})
    return spark.createDataFrame(pdf)

df = _crear_datos(int(os.environ.get("DATA_N", "200000")))
ESQUEMA = ", ".join([f"{c} ({t})" for c, t in df.dtypes])

# ---------- 2) El LLM que escribe el codigo (GLM de Z.ai) ----------
MODELO_LLM = "glm-4.5-flash"   # variante Flash (capa gratuita); puede ajustarse (ej. "glm-4.7-flash")

_cliente = None
def _get_cliente():
    """Crea el cliente (OpenAI -> Z.ai) la primera vez que se usa (lee ZAI_API_KEY del entorno)."""
    global _cliente
    if _cliente is None:
        _cliente = OpenAI(api_key=os.environ.get("ZAI_API_KEY", ""),
                          base_url="https://api.z.ai/api/paas/v4")
    return _cliente

INSTRUCCION = f"""Eres un analista de datos experto en PySpark.
Existe un DataFrame de Spark llamado df con estas columnas: {ESQUEMA}.
Escribe codigo PySpark que responda la pregunta del usuario.
Reglas estrictas:
- Usa unicamente df, spark y F (pyspark.sql.functions). No leas ni escribas archivos.
- Asigna SIEMPRE el resultado final a una variable llamada resultado.
- Si el resultado es una tabla, deja resultado como un DataFrame de Spark (no uses .show()).
- Devuelve SOLO el codigo Python, sin explicaciones ni bloques de markdown."""

def generar_codigo(pregunta):
    """Pide al modelo (GLM) el codigo PySpark para la pregunta y limpia la respuesta."""
    r = _get_cliente().chat.completions.create(
        model=MODELO_LLM,
        messages=[{"role": "system", "content": INSTRUCCION},
                  {"role": "user", "content": pregunta}])
    codigo = r.choices[0].message.content.strip()
    if codigo.startswith("```"):
        codigo = codigo.strip("`")
        if codigo.lstrip().startswith("python"):
            codigo = codigo.lstrip()[len("python"):]
    return codigo.strip()

# ---------- 3) Ejecutar el codigo generado ----------
def ejecutar(codigo):
    """Ejecuta el codigo en un entorno controlado y devuelve 'resultado' como tabla."""
    entorno = {"spark": spark, "df": df, "F": F, "resultado": None}
    exec(codigo, entorno)
    res = entorno["resultado"]
    if res is None:
        return None
    if hasattr(res, "toPandas"):
        return res.limit(50).toPandas()
    return pd.DataFrame({"resultado": [res]})

# ---------- 4) Unir todo: pregunta -> codigo -> resultado ----------
def analizar(pregunta):
    if not pregunta.strip():
        return "", None, None
    try:
        codigo = generar_codigo(pregunta)
    except Exception as e:
        return f"# Error al generar el codigo: {e}", None, None
    try:
        tabla = ejecutar(codigo)
    except Exception as e:
        return codigo, pd.DataFrame({"Error": [str(e)]}), None
    fig = None
    if tabla is not None and tabla.shape[1] == 2 and pd.api.types.is_numeric_dtype(tabla.iloc[:, 1]):
        fig, ax = plt.subplots(figsize=(7, 3.5))
        ax.bar(tabla.iloc[:, 0].astype(str), tabla.iloc[:, 1], color="#FFD21E", edgecolor="white")
        ax.set_ylabel(tabla.columns[1]); plt.xticks(rotation=30, ha="right"); plt.tight_layout()
    return codigo, tabla, fig

# ---------- 5) Interfaz web ----------
EJEMPLOS = ["¿Cuál es la tarifa media por tipo de pago?",
            "Muestra el ingreso total por zona, de mayor a menor",
            "¿A qué hora del día la tarifa media es más alta?",
            "¿Cuántos viajes hubo en fin de semana frente a entre semana?"]

with gr.Blocks(title="Analista de datos con IA") as demo:
    gr.Markdown("# Analista de datos con IA y Spark\n"
                "Escribe una pregunta en lenguaje natural. La IA escribe el análisis en PySpark, se ejecuta sobre los datos y se muestra el resultado.")
    pregunta = gr.Textbox(label="Tu pregunta", placeholder="¿Cuál es la tarifa media por zona?")
    btn = gr.Button("Analizar", variant="primary")
    gr.Examples(EJEMPLOS, inputs=pregunta)
    codigo_out = gr.Code(label="Código PySpark que escribió la IA", language="python")
    tabla_out = gr.Dataframe(label="Resultado")
    chart_out = gr.Plot(label="Gráfico (si aplica)")
    btn.click(analizar, pregunta, [codigo_out, tabla_out, chart_out])

if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", server_port=7860)

Overwriting app.py


## Paso 4b — Prueba local de la aplicación
Se ejecuta la app dentro de Colab para verla funcionar. `share=True` genera un enlace público temporal. (Para continuar, detener la celda.)

In [8]:
import importlib, app as app_module
importlib.reload(app_module)
app_module.demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://42fcb1475b5ebd0478.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Paso 5 — Empaquetado para el despliegue

In [9]:
import pyspark, gradio, pandas, numpy, matplotlib, openai
# Se anotan las versiones exactas para que el contenedor reproduzca este entorno
reqs = f"""pyspark=={pyspark.__version__}
gradio=={gradio.__version__}
openai=={openai.__version__}
pandas=={pandas.__version__}
numpy=={numpy.__version__}
matplotlib=={matplotlib.__version__}
"""
open("requirements.txt","w").write(reqs); print(reqs)

pyspark==4.1.2
gradio==6.17.3
openai==2.43.0
pandas==2.2.2
numpy==2.0.2
matplotlib==3.10.0



In [10]:
%%writefile Dockerfile
FROM python:3.11-slim
# Spark necesita Java: se instala el kit de desarrollo de Java
RUN apt-get update && apt-get install -y default-jdk && rm -rf /var/lib/apt/lists/*
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py .
EXPOSE 7860
CMD ["python", "app.py"]

Overwriting Dockerfile


In [11]:
%%writefile README.md
---
title: Analista de datos con IA
emoji: 🤖
colorFrom: yellow
colorTo: indigo
sdk: docker
app_port: 7860
pinned: false
---
# Analista de datos con IA y Spark
Pregunta en lenguaje natural; la IA escribe el análisis en PySpark y lo ejecuta.

Overwriting README.md


## Paso 6 — Despliegue en Hugging Face Spaces
Igual que en la clase 3, pero con un detalle: la API key de Gemini se sube como **secret** del Space (no va dentro del código). El contenedor instala Java + Spark, por lo que la construcción tarda algunos minutos más.

In [12]:
from getpass import getpass
from huggingface_hub import HfApi, create_repo, add_space_secret

HF_TOKEN = getpass("Token de Hugging Face (Write): ")
USUARIO  = input("Usuario de Hugging Face: ").strip()
SPACE_ID = f"{USUARIO}/analista-datos-ia"

create_repo(SPACE_ID, repo_type="space", space_sdk="docker", token=HF_TOKEN, exist_ok=True)
api = HfApi(token=HF_TOKEN)

# Se guarda la clave de Z.ai como secret del Space (la app la lee desde ZAI_API_KEY)
add_space_secret(SPACE_ID, "ZAI_API_KEY", os.environ["ZAI_API_KEY"], token=HF_TOKEN)

for archivo in ["app.py", "requirements.txt", "Dockerfile", "README.md"]:
    api.upload_file(path_or_fileobj=archivo, path_in_repo=archivo, repo_id=SPACE_ID, repo_type="space")
    print("subido:", archivo)

print(f"\nDesplegando. La construcción (Java + Spark) tarda varios minutos.")
print(f"Estado del Space: https://huggingface.co/spaces/{SPACE_ID}")



Token de Hugging Face (Write): ··········
Usuario de Hugging Face: lcherrerap


No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


subido: app.py
subido: requirements.txt


No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


subido: Dockerfile
subido: README.md

Desplegando. La construcción (Java + Spark) tarda varios minutos.
Estado del Space: https://huggingface.co/spaces/lcherrerap/analista-datos-ia


---
## Cierre del curso

Esta aplicación reúne todo el recorrido del curso:

- **Spark** (clase 1): ejecuta el análisis y es el camino hacia Big Data.
- **Despliegue** (clase 3): publicar una app con su contenedor en Hugging Face.
- **IA generativa**: el LLM convierte el lenguaje natural en análisis.

### Actividad
1. Cambiar el dataset por uno propio y ajustar `_crear_datos()` (o cargar un CSV).
2. Probar preguntas más difíciles y observar el código que escribe la IA.
3. Desplegar la app y entregar la URL pública.

### Idea para Big Data real
El mismo código PySpark que genera la IA funcionaría, sin cambios, sobre un clúster con terabytes (por ejemplo, en Databricks). Aquí corre sobre un dataset mediano, pero el patrón es el que escala.